In [1]:
# !pip install yfinance pandas FinMind tqdm requests


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from FinMind.data import DataLoader
import warnings
import os
from pathlib import Path
from io import StringIO
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError

warnings.filterwarnings('ignore')

# FINRA short-volume 需要逐交易日下載，2017-2026 約兩千多個 request；
# 預設跳過，避免整個資料 notebook 跑數小時。需要美股融券 proxy 時再改 True。
FETCH_FINRA_SHORT_VOLUME = False

US_TARGET_PROXY = {
    "TSM": "TSM",
    "^GSPC": "SPY",
    "^NDX": "QQQ",
    "^SOX": "SOXX",
}

US_FUTURES_PROXY = {
    "TSM": "NQ=F",
    "^GSPC": "ES=F",
    "^NDX": "NQ=F",
    "^SOX": "NQ=F",
}


def resolve_output_dir():
    return Path("data/taiwan_market_data") if Path("data").exists() else Path("taiwan_market_data")


def fetch_finra_short_volume(start_date, end_date):
    """用 FINRA Reg SHO daily short volume 建立美股融券對應資料。

    這不是台灣的融資融券餘額；美股沒有同樣的日頻公開欄位。
    這裡保留 event_combo.py 的 margin schema，將 ShortSale* 欄位填入 daily short volume，
    MarginPurchase* 欄位留 NaN，避免偽造不存在的融資資料。
    """
    rows = []
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)
    symbols = sorted(set(US_TARGET_PROXY.values()))
    bdays = pd.date_range(start, end, freq="B")

    for date in bdays:
        ymd = date.strftime("%Y%m%d")
        url = f"https://cdn.finra.org/equity/regsho/daily/CNMSshvol{ymd}.txt"
        try:
            req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(req, timeout=8) as resp:
                text = resp.read().decode("utf-8", errors="ignore")
        except (HTTPError, URLError, TimeoutError):
            continue

        try:
            raw = pd.read_csv(StringIO(text), sep="|")
        except Exception:
            continue
        if raw.empty or "Symbol" not in raw.columns:
            continue

        raw = raw[raw["Symbol"].isin(symbols)].copy()
        if raw.empty:
            continue
        grouped = raw.groupby("Symbol", as_index=False).agg({
            "ShortVolume": "sum",
            "ShortExemptVolume": "sum",
            "TotalVolume": "sum",
        })

        for target, proxy in US_TARGET_PROXY.items():
            hit = grouped[grouped["Symbol"] == proxy]
            if hit.empty:
                continue
            r = hit.iloc[0]
            short_volume = float(r.get("ShortVolume", np.nan))
            total_volume = float(r.get("TotalVolume", np.nan))
            short_exempt = float(r.get("ShortExemptVolume", 0.0))
            rows.append({
                "date": date.date().isoformat(),
                "stock_id": target,
                "proxy_symbol": proxy,
                "MarginPurchaseTodayBalance": np.nan,
                "ShortSaleTodayBalance": short_volume,
                "MarginPurchaseBuy": np.nan,
                "MarginPurchaseSell": np.nan,
                "ShortSaleBuy": short_volume,
                "ShortSaleSell": max(total_volume - short_volume, 0.0) if np.isfinite(total_volume) else np.nan,
                "ShortExemptVolume": short_exempt,
                "TotalVolume": total_volume,
                "ShortVolumeRatio": short_volume / total_volume if total_volume else np.nan,
            })

    return pd.DataFrame(rows)


def build_us_futures_night(open_df, price_df, volume_df):
    """建立美股夜盤對應資料。

    台股使用 FinMind 的 TX after_market；美股用 Yahoo continuous futures 的 overnight gap
    作為同語意 proxy：今日 futures open 相對前一日 futures close 的變化。
    """
    rows = []
    for target, proxy in US_FUTURES_PROXY.items():
        if proxy not in open_df.columns or proxy not in price_df.columns:
            print(f"   [警告] 找不到 {target} 的 futures proxy {proxy}，略過夜盤資料。")
            continue
        prev_close = price_df[proxy].shift(1).astype(float)
        today_open = open_df[proxy].astype(float)
        spread = today_open - prev_close
        spread_per = today_open / prev_close - 1
        volume = volume_df[proxy].replace(0, np.nan).astype(float) if proxy in volume_df.columns else np.nan
        tmp = pd.DataFrame({
            "date": price_df.index,
            "stock_id": target,
            "proxy_futures": proxy,
            "spread": spread.values,
            "spread_per": spread_per.values,
            "volume": volume.values if hasattr(volume, "values") else volume,
        })
        rows.append(tmp)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def empty_us_institutional_schema():
    """保留美股法人資料接口，不用 volume proxy 偽造三大法人。

    若你後續取得每日美股法人流資料，輸出同 schema 後 event_combo.py 會自動讀取：
    date, stock_id, name, buy, sell。
    """
    return pd.DataFrame(columns=["date", "stock_id", "name", "buy", "sell"])


def fetch_predictive_modeling_data(start_date="2023-01-01", end_date="2024-01-01"):
    print("初始化資料下載模組...")
    
    tickers = [
        "2330.TW", "3711.TW", "2382.TW", "2454.TW", "0050.TW", 
        "2317.TW", "2308.TW", "2303.TW", "2376.TW", "2377.TW", 
        "00632R.TW",
        "TSM", "^SOX", "^NDX", "^GSPC", "^VIX", 
        "TWD=X", "^TNX",
        "SPY", "QQQ", "SOXX", "ES=F", "NQ=F"
    ]
    
    print("=> 正在從 Yahoo Finance 下載全球市場資料...")
    yf_raw = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)
    price_df = yf_raw['Close']
    volume_df = yf_raw['Volume']
    open_df = yf_raw['Open']
    price_df.ffill(inplace=True)
    volume_df.ffill(inplace=True)
    open_df.ffill(inplace=True)

    print("=> 正在建立美股融券與夜盤對應資料...")
    if FETCH_FINRA_SHORT_VOLUME:
        us_margin_df = fetch_finra_short_volume(start_date, end_date)
    else:
        print("   -> 跳過 FINRA short-volume 逐日下載；輸出空的美股融券 schema。")
        us_margin_df = pd.DataFrame(columns=[
            "date", "stock_id", "proxy_symbol",
            "MarginPurchaseTodayBalance", "ShortSaleTodayBalance",
            "MarginPurchaseBuy", "MarginPurchaseSell",
            "ShortSaleBuy", "ShortSaleSell",
            "ShortExemptVolume", "TotalVolume", "ShortVolumeRatio",
        ])
    us_futures_night_df = build_us_futures_night(open_df, price_df, volume_df)
    us_institutional_df = empty_us_institutional_schema()

    print("=> 正在從 FinMind 獲取三大法人與籌碼資料...")
    api = DataLoader()
    target_stocks = [
        '2330', '3711', '2382', '2454', '0050', 
        '2317', '2308', '2303', '2376', '2377', 
        '00632R', '00679B'
    ]
    all_institutional = []
    all_margin = []
    
    for stock_id in target_stocks:
        print(f"   正在下載 {stock_id} 的法人與融資券資料...")
        inst_df = api.taiwan_stock_institutional_investors(
            stock_id=stock_id, start_date=start_date, end_date=end_date
        )
        if not inst_df.empty:
            all_institutional.append(inst_df)
            
        mar_df = api.taiwan_stock_margin_purchase_short_sale(
            stock_id=stock_id, start_date=start_date, end_date=end_date
        )
        if not mar_df.empty:
            all_margin.append(mar_df)
            
    institutional_df = pd.concat(all_institutional, ignore_index=True) if all_institutional else pd.DataFrame()
    margin_df = pd.concat(all_margin, ignore_index=True) if all_margin else pd.DataFrame()

    print("=> 正在獲取台指期 (TX) 夜盤資料...")
    futures_df = api.taiwan_futures_daily(
        futures_id="TX", start_date=start_date, end_date=end_date
    )
    tx_night_df = pd.DataFrame()
    if not futures_df.empty:
        if 'trading_session' in futures_df.columns:
            all_night_data = futures_df[futures_df['trading_session'] == 'after_market']
            if not all_night_data.empty:
                idx = all_night_data.groupby('date')['volume'].idxmax()
                tx_night_df = all_night_data.loc[idx].sort_values('date')
                print(f"   -> 成功篩選出最具代表性的夜盤近月合約資料，共 {len(tx_night_df)} 筆。")
        else:
            print(f"   [警告] 找不到 'trading_session' 欄位。目前的欄位為: {list(futures_df.columns)}")

    print("資料下載與整合完成！\n")
    return (
        price_df,
        volume_df,
        open_df,
        institutional_df,
        margin_df,
        tx_night_df,
        us_institutional_df,
        us_margin_df,
        us_futures_night_df,
    )


if __name__ == "__main__":
    START_DATE = "2017-01-01"
    END_DATE = "2026-05-18"
    
    (
        prices,
        volumes,
        opens,
        inst_data,
        margin_data,
        tx_night,
        us_inst_data,
        us_margin_data,
        us_futures_night,
    ) = fetch_predictive_modeling_data(start_date=START_DATE, end_date=END_DATE)

    output_dir = resolve_output_dir()
    output_dir.mkdir(parents=True, exist_ok=True)

    print("=> 正在將資料寫入 CSV 檔案...")
    prices.to_csv(output_dir / "global_prices.csv", index=True)
    print(f"   [已儲存] 價格資料 -> {output_dir / 'global_prices.csv'}")
    volumes.to_csv(output_dir / "global_volumes.csv", index=True)
    print(f"   [已儲存] 成交量資料 -> {output_dir / 'global_volumes.csv'}")
    opens.to_csv(output_dir / "global_opens.csv", index=True)
    print(f"   [已儲存] 開盤價資料 -> {output_dir / 'global_opens.csv'}")
    
    if not inst_data.empty:
        inst_data.to_csv(output_dir / "institutional_investors.csv", index=False)
        print(f"   [已儲存] 台股三大法人籌碼 -> {output_dir / 'institutional_investors.csv'}")
    if not margin_data.empty:
        margin_data.to_csv(output_dir / "margin_trading.csv", index=False)
        print(f"   [已儲存] 台股融資融券餘額 -> {output_dir / 'margin_trading.csv'}")
    if not tx_night.empty:
        tx_night.to_csv(output_dir / "tx_futures_night.csv", index=False)
        print(f"   [已儲存] 台指期夜盤數據 -> {output_dir / 'tx_futures_night.csv'}")

    us_inst_data.to_csv(output_dir / "us_institutional_investors.csv", index=False)
    print(f"   [已儲存] 美股法人資料接口 -> {output_dir / 'us_institutional_investors.csv'}")
    us_margin_data.to_csv(output_dir / "us_margin_trading.csv", index=False)
    print(f"   [已儲存] 美股融券對應資料 schema -> {output_dir / 'us_margin_trading.csv'}")
    if not us_futures_night.empty:
        us_futures_night.to_csv(output_dir / "us_futures_night.csv", index=False)
        print(f"   [已儲存] 美股夜盤對應資料 futures overnight gap -> {output_dir / 'us_futures_night.csv'}")
        
    print(f"\n所有市場數據已成功儲存至 '{output_dir}/' 資料夾下。")


初始化資料下載模組...
=> 正在從 Yahoo Finance 下載全球市場資料...


[*********************100%***********************]  23 of 23 completed


=> 正在建立美股融券與夜盤對應資料...
   -> 跳過 FINRA short-volume 逐日下載；輸出空的美股融券 schema。
=> 正在從 FinMind 獲取三大法人與籌碼資料...


2026-06-01 17:26:02.963 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-06-01 17:26:02.964 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2330


   正在下載 2330 的法人與融資券資料...


...

   正在下載 00632R 的法人與融資券資料...


2026-06-01 17:26:06.626 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 00632R
2026-06-01 17:26:07.353 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 00679B


   正在下載 00679B 的法人與融資券資料...


2026-06-01 17:26:08.432 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 00679B
2026-06-01 17:26:08.962 | INFO     | FinMind.data.finmind_api:get_data:164 - download Dataset.TaiwanFuturesDaily, data_id: TX


=> 正在獲取台指期 (TX) 夜盤資料...
   -> 成功篩選出最具代表性的夜盤近月合約資料，共 2196 筆。
資料下載與整合完成！

=> 正在將資料寫入 CSV 檔案...
   [已儲存] 價格資料 -> taiwan_market_data/global_prices.csv
   [已儲存] 成交量資料 -> taiwan_market_data/global_volumes.csv
   [已儲存] 開盤價資料 -> taiwan_market_data/global_opens.csv
   [已儲存] 台股三大法人籌碼 -> taiwan_market_data/institutional_investors.csv
   [已儲存] 台股融資融券餘額 -> taiwan_market_data/margin_trading.csv
   [已儲存] 台指期夜盤數據 -> taiwan_market_data/tx_futures_night.csv
   [已儲存] 美股法人資料接口 -> taiwan_market_data/us_institutional_investors.csv
   [已儲存] 美股融券對應資料 schema -> taiwan_market_data/us_margin_trading.csv
   [已儲存] 美股夜盤對應資料 futures overnight gap -> taiwan_market_data/us_futures_night.csv

所有市場數據已成功儲存至 'taiwan_market_data/' 資料夾下。
